# LLMBackend FrameNet Graph Queries

This notebook resolves one pickup instruction, projects the FrameNet sidecar into `HypothesisGraph`, then inspects the result through the current **family/view** API:

- `FrameNetFamily.make_manager(...)` wires projector orchestration into `LLMBackend`
- `FrameNetFamily.make_view(...)` exposes typed FrameNet query domains over the generic graph
- EQL queries still operate on normal node domains like `FrameHypothesisNode` and `FrameRoleHypothesisNode`
- graph-native helpers such as `subgraph_for_action(...)`, `edge_exists(...)`, and DOT visualization remain available


In [ ]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body
print("World loaded:", type(world).__name__)
print("Robot:", robot_view)


In [ ]:
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))


## FrameNet hypothesis-graph setup

The next cell creates a fresh `HypothesisGraph`, then uses the current family abstraction to build both sides of the workflow:

- `FrameNetFamily.make_manager(graph)` for automatic projection during backend evaluation
- `FrameNetFamily.make_view(graph)` for FrameNet-specific query access over the generic graph


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr.backend import LLMBackend
from llmr.hypotheses import (
    AboutActionEdge,
    ClaimStatus,
    EvokesFrameEdge,
    FrameHypothesisNode,
    FrameNetFamily,
    FrameRoleHypothesisNode,
    GroundedByEdge,
    GroundingState,
    HasRoleEdge,
    HypothesisGraph,
    ProducedClaimEdge,
    SlotBindingEvidenceNode,
    SupportedByEdge,
    SymbolGroundingEvidenceNode,
)
from llmr.reasoning.framenet_reasoner import FrameNetReasoner
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction

graph = HypothesisGraph()
manager = FrameNetFamily.make_manager(graph)
framenet = FrameNetFamily.make_view(graph)

INSTRUCTION = "pick up the milk from the table"


def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )


def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction=instruction,
        strict_required=True,
        reasoners=[FrameNetReasoner(llm=llm)],
        hypothesis_graph_manager=manager,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend


def role_rows(roles):
    return [
        {
            "display_id": role.display_id,
            "role": role.role_name,
            "family": role.role_family,
            "text": role.filler_text,
            "kind": role.filler_kind,
            "status": role.meta.status.value,
            "grounding": role.meta.grounding.value,
            "run_id": role.meta.short_run_id,
        }
        for role in roles
    ]


print("Graph and helpers ready.")


In [ ]:
action, backend = run_instruction(INSTRUCTION)
current_frame = framenet.frames()[-1]
current_run_id = current_frame.meta.run_id
action_view = framenet.action_subgraph(action)
run_view = framenet.run_subgraph(current_run_id)
action_graph = action_view.graph
run_graph = run_view.graph

print("Resolved action:", type(action).__name__)
print("Resolved object:", action.object_designator)
print("Resolved arm:", action.arm)
print("Run id:", current_run_id)
print("Short run id:", current_frame.meta.short_run_id)
print("Current frame id:", current_frame.display_id)
print("Full graph:", graph.node_count, "nodes /", graph.edge_count, "edges")
print("Action subgraph:", action_graph.node_count, "nodes /", action_graph.edge_count, "edges")
print("Run subgraph:", run_graph.node_count, "nodes /", run_graph.edge_count, "edges")


In [ ]:
# backend.semantics

In [ ]:
import json

print(json.dumps(backend.semantics.frames.model_dump(by_alias=True), indent=2, default=str))


## Graph-native inspection

Before switching to EQL, inspect the graph directly. The next cell shows how the new graph helpers expose the same hypothesis structure without going through EQL domains first.


In [ ]:
theme_candidates = [
    node
    for node in action_view.roles_by_role_name("theme")
    if node.meta.run_id == current_run_id
]
theme_role = theme_candidates[0]

{
    "frame_id": current_frame.display_id,
    "theme_role_id": theme_role.display_id,
    "frame_targets": [node.display_id for node in action_graph.get_targets(current_frame.id, HasRoleEdge)],
    "theme_incoming_edges": [edge.display_id for edge in action_graph.in_edges(theme_role.id, HasRoleEdge)],
    "has_theme_relation": action_graph.edge_exists(
        current_frame.id,
        theme_role.id,
        HasRoleEdge,
        run_id=current_run_id,
    ),
}


## Visualization

Render the action-local hypothesis graph. If `graphviz` is installed, the DOT graph is displayed inline; otherwise the DOT source is printed so you can still inspect the structure.


In [ ]:
try:
    from graphviz import Source
    from IPython.display import display

    display(Source(action_graph.to_dot(), format="svg"))
except Exception as exc:
    print("Graphviz rendering unavailable:", exc)
    print(action_graph.to_dot())


## Query setup

These EQL variables use the action-local `FrameNetGraphView` as their domain source. That keeps the graph core generic while still giving this notebook typed FrameNet access. The later queries still join through typed edges like `HasRoleEdge`, `SupportedByEdge`, and `GroundedByEdge`.


In [ ]:
from krrood.entity_query_language.factories import an, entity, variable

frame = variable(FrameHypothesisNode, domain=action_view.frames())
role = variable(FrameRoleHypothesisNode, domain=action_view.roles())
has_role = variable(HasRoleEdge, domain=action_graph.edge_domain(HasRoleEdge))
support = variable(SlotBindingEvidenceNode, domain=action_graph.domain(SlotBindingEvidenceNode))
support_edge = variable(SupportedByEdge, domain=action_graph.edge_domain(SupportedByEdge))
grounding = variable(SymbolGroundingEvidenceNode, domain=action_graph.domain(SymbolGroundingEvidenceNode))
grounding_edge = variable(GroundedByEdge, domain=action_graph.edge_domain(GroundedByEdge))

{
    "frames": [item.display_id for item in action_view.frames()],
    "roles": [item.display_id for item in action_view.roles()],
}


In [ ]:
# Query 1: Frame hypothesis for the exact instruction
frame_query = an(
    entity(frame).where(
        frame.instruction_text == INSTRUCTION,
        frame.meta.run_id == current_run_id,
    )
)
frame_results = list(frame_query.evaluate())
frame_result = frame_results[0]

[
    {
        "display_id": item.display_id,
        "frame": item.frame,
        "lexical_unit": item.lexical_unit,
        "label": item.framenet_label,
        "status": item.meta.status.value,
        "grounding": item.meta.grounding.value,
        "run_id": item.meta.short_run_id,
    }
    for item in frame_results
]


In [ ]:
# Query 1: Frame hypothesis for the exact instruction
frame_query = an(
    entity(frame).where(
        frame.instruction_text == INSTRUCTION,
        frame.meta.run_id == current_run_id,
    )
)
frame_results = list(frame_query.evaluate())
frame_result = frame_results[0]

[
    {
        "frame": item.frame,
        "lexical_unit": item.lexical_unit,
        "label": item.framenet_label,
        "status": item.meta.status.value,
        "grounding": item.meta.grounding.value,
        "run_id": item.meta.run_id,
    }
    for item in frame_results
]


In [ ]:
# Query 2: Role hypotheses attached to that frame through HasRoleEdge
frame_roles_query = an(
    entity(role).where(
        has_role.src_id == frame_result.id,
        has_role.dst_id == role.id,
    )
)
frame_roles = list(frame_roles_query.evaluate())
role_rows(frame_roles)


In [ ]:
# Query 3: Grounded roles only
grounded_roles_query = an(
    entity(role).where(
        role.meta.run_id == current_run_id,
        role.meta.grounding == GroundingState.SYMBOL_GROUNDED,
    )
)
grounded_roles = list(grounded_roles_query.evaluate())
role_rows(grounded_roles)


In [ ]:
# Query 4: Hypothesis-only roles (not grounded)
hypothesis_only_query = an(
    entity(role).where(
        role.meta.run_id == current_run_id,
        role.meta.status == ClaimStatus.HYPOTHESIS,
    )
)
hypothesis_only_roles = list(hypothesis_only_query.evaluate())
role_rows(hypothesis_only_roles)


In [ ]:
# Query 6: Slot-support evidence attached to supported roles
slot_support_query = an(
    entity(support).where(
        support_edge.dst_id == support.id,
        support_edge.src_id == role.id,
        role.meta.run_id == current_run_id,
    )
)
slot_support_results = list(slot_support_query.evaluate())

[
    {
        "display_id": item.display_id,
        "slot_name": item.slot_name,
        "value_repr": item.value_repr,
        "run_id": item.meta.short_run_id,
    }
    for item in slot_support_results
]


In [ ]:
# Query 7: Symbol grounding evidence attached to grounded roles
grounding_query = an(
    entity(grounding).where(
        grounding_edge.dst_id == grounding.id,
        grounding_edge.src_id == role.id,
        role.meta.run_id == current_run_id,
    )
)
grounding_results = list(grounding_query.evaluate())

[
    {
        "display_id": item.display_id,
        "query_text": item.query_text,
        "symbol_type": item.symbol_type,
        "grounding_method": item.grounding_method,
        "symbol": repr(item.symbol_ref),
    }
    for item in grounding_results
]


In [ ]:
# Query 7: Symbol grounding evidence attached to grounded roles
grounding_query = an(
    entity(grounding).where(
        grounding_edge.dst_id == grounding.id,
        grounding_edge.src_id == role.id,
        role.meta.run_id == current_run_id,
    )
)
grounding_results = list(grounding_query.evaluate())

[
    {
        "query_text": item.query_text,
        "symbol_type": item.symbol_type,
        "grounding_method": item.grounding_method,
        "symbol": repr(item.symbol_ref),
    }
    for item in grounding_results
]


## Try your own instruction

To test a different pickup instruction:

1. Re-run the setup cell that creates `graph`, `registry`, and `manager` so the graph is empty again.
2. Change `INSTRUCTION`.
3. Re-run the execution cell, the graph inspection cells, and the query cells.

The notebook is intentionally single-run per reset so the action-local subgraph and the EQL domains stay easy to inspect.
